# Entrenament del Truc (etapa `cartes`) a Google Colab

**Factors importants abans d'executar:**

1. **Cal haver pujat els canvis locals a GitHub** (`git push`). Aquest notebook clona el repo des d'`origin`, no des del teu disc — si hi ha canvis sense pujar (redisseny d'obs, `AgentRegles` recalibrat, `AgentProbabilistic`, `target_kl`/`--resume_from`...), Colab no els veurà.
2. **Tria l'entorn d'execució CPU, no GPU**: Entorn d'execució → Canvia el tipus d'entorn d'execució → "Cap acceleració". Aquesta feina és de CPU (l'entorn del joc, no la xarxa), la GPU no ajuda i només gasta la quota limitada.
3. **Colab gratuït es desconnecta** (~90 min d'inactivitat, sessió màxima ~12h). Els checkpoints es guarden a Google Drive perquè sobrevisquin la desconnexió, i aquest notebook detecta sol si ja n'hi ha per continuar (`--resume_from`) en lloc de començar de zero cada vegada. **Quan es desconnecti, torna a obrir el notebook i executa totes les cel·les de nou.**
4. **Serà més lent que en local**: Colab gratuït sol donar ~2 vCPUs, molt menys que els 8 cores locals amb què s'han fet totes les proves d'aquesta sessió. No és una manera de fer-ho més ràpid, només de no bloquejar el teu ordinador.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuració

`REPO_URL` ha de ser accessible (públic, o privat amb credencials configurades a Colab). `DRIVE_DIR` és on es clona el projecte dins de Google Drive (persistent entre sessions, així no cal re-clonar cada cop).

In [ ]:
REPO_URL = "https://github.com/JoFeF08/TFG-truc.git"
DRIVE_DIR = "/content/drive/MyDrive/tfg-truc"
BRANCH = "master"

In [ ]:
import os

if not os.path.exists(DRIVE_DIR):
    !git clone --branch {BRANCH} {REPO_URL} "{DRIVE_DIR}"
else:
    %cd {DRIVE_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {DRIVE_DIR}
!git log -1 --oneline

## Dependències

Torch en versió CPU (sense CUDA) — la feina no fa servir GPU, i la roda CUDA és molt més gran i lenta de descarregar.

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q stable-baselines3 sb3-contrib gymnasium pettingzoo tqdm

In [ ]:
import os
n_cpus = os.cpu_count()
NUM_ENVS = max(1, n_cpus)
print(f"CPUs disponibles a aquesta màquina de Colab: {n_cpus} -> --num_envs {NUM_ENVS}")

## Represa automàtica

Els checkpoints es guarden a `POOL_DIR`, dins de Google Drive (sobreviuen la desconnexió). Si ja n'hi ha algun, es reprèn l'entrenament en lloc de començar de zero.

In [ ]:
from pathlib import Path

POOL_DIR = f"{DRIVE_DIR}/RL/entrenament/registres_colab/pool"
Path(POOL_DIR).mkdir(parents=True, exist_ok=True)

checkpoints = sorted(
    Path(POOL_DIR).glob("*_steps.zip"),
    key=lambda p: int(p.stem.rsplit('_', 2)[-2]),
)
if checkpoints:
    print(f"Checkpoint previ trobat: {checkpoints[-1].name} -- es reprendrà des d'aquí.")
else:
    print("Cap checkpoint previ -- entrenament nou des de zero.")

## Entrenament

Mateixa configuració que en local (`--opponent fort` amb `AgentProbabilistic`, `--reward_scale 15 --ent_coef 0.01`), amb `--target_kl 0.03` afegit (l'entrenament local va mostrar `approx_kl`/`clip_fraction` sostingudament alts amb `reward_scale=15` -- `target_kl` fa que PPO aturi cada època d'actualització si el KL el supera, evitant la divergència).

In [ ]:
cmd = (
    f'python RL/entrenament/entrenament_sb3.py '
    f'--stage cartes --opponent fort '
    f'--reward_scale 15 --ent_coef 0.01 --target_kl 0.03 '
    f'--num_envs {NUM_ENVS} --total_timesteps 12000000 '
    f'--pool_dir "{POOL_DIR}"'
)
if checkpoints:
    cmd += f' --resume_from "{POOL_DIR}"'

print(cmd)
!{cmd}

## Si es desconnecta la sessió

Torna a obrir aquest notebook i executa totes les cel·les de dalt a baix (`Entorn d'execució` → `Executa-ho tot`). La cel·la de "Represa automàtica" detectarà el darrer checkpoint a `POOL_DIR` (a Google Drive, no es perd) i la cel·la d'entrenament hi continuarà amb `--resume_from` en lloc de començar de nou.

## (Opcional) Avaluació ràpida del darrer checkpoint

In [ ]:
checkpoints = sorted(
    Path(POOL_DIR).glob("*_steps.zip"),
    key=lambda p: int(p.stem.rsplit('_', 2)[-2]),
)
if checkpoints:
    darrer = checkpoints[-1]
    print(f"Avaluant: {darrer}")
    !python -m RL.tools.avaluacio_duplicada --model "{darrer}" --rival probabilistic --n_repartiments 100
    !python -m RL.tools.avaluacio_duplicada --model "{darrer}" --rival agressiu --n_repartiments 100
else:
    print("Encara no hi ha cap checkpoint.")